In [ ]:
# Check if running on Colab
import sys
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    print("[INFO] Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("[INFO] Running locally")

# Setup environment
if IN_COLAB:
    # Mount Google Drive
    drive.mount('/content/drive', force_remount=False)
    print("[OK] Google Drive mounted")

    # Clone repository
    print("[Clone] Cloning repository...")
    !git clone -b develop https://github.com/senkochi/taxi-demand-prediction.git /content/taxi-demand-prediction 2>/dev/null || echo "Repository already cloned"

    os.chdir('/content/taxi-demand-prediction')
    print(f"[OK] Working directory: {os.getcwd()}")

    # Link data from Google Drive
    print("\n[Link] Linking data from Google Drive...")
    drive_data = Path('/content/drive/MyDrive/data')
    local_data = Path('data')

    if not local_data.exists():
        try:
            os.symlink(drive_data, 'data')
            print(f"  [OK] Symlink: ./data → {drive_data}")
        except (OSError, NotImplementedError):
            import shutil
            shutil.copytree(drive_data, 'data')
            print(f"  [OK] Data copied from Google Drive")
    else:
        print(f"  [OK] ./data already exists")
else:
    # Local setup - navigate to project root
    notebook_dir = Path.cwd()

    # If in notebooks folder, go up to project root
    if notebook_dir.name == 'notebooks':
        project_root = notebook_dir.parent
    elif (notebook_dir / 'notebooks').exists():
        project_root = notebook_dir
    else:
        project_root = notebook_dir.parent

    os.chdir(project_root)
    print(f"[OK] Working directory: {os.getcwd()}")

# Verify structure
print("\n[Check] Project structure:")
for item in ['src', 'config', 'scripts', 'data']:
    exists = Path(item).exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {item}/")

[INFO] Running on Google Colab
Mounted at /content/drive
[OK] Google Drive mounted
[Clone] Cloning repository...
[OK] Working directory: /content/taxi-demand-prediction

[Link] Linking data from Google Drive...
  [OK] Symlink: ./data → /content/drive/MyDrive/data

[Check] Project structure:
  ✓ src/
  ✓ config/
  ✓ scripts/
  ✓ data/


In [ ]:
# Install dependencies on Colab
if IN_COLAB:
    print("[Install] Installing PyTorch Lightning...")
    !pip install pytorch-lightning -q
    !pip install duckdb -q
    !pip install pyyaml -q
    print("[OK] Dependencies installed")

[Install] Installing PyTorch Lightning...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 40.0 MB/s eta 0:00:00
[OK] Dependencies installed


In [ ]:
# Check GPU availability
import torch

print("[System] PyTorch environment:")
print(f"  - Version: {torch.__version__}")
print(f"  - CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print(f"  - Using CPU")
    device = 'cpu'

[System] PyTorch environment:
  - Version: 2.11.0+cpu
  - CUDA available: False
  - Using CPU


In [ ]:
import yaml
from pathlib import Path
import os

# Get actual current working directory
cwd = os.getcwd()
print(f"[Debug] Current working directory: {cwd}\n")

# Try multiple possible config paths
config_paths = [
    Path('config/config.yaml'),  # Relative to cwd
    Path.cwd() / 'config' / 'config.yaml',  # Absolute from cwd
]

# If running from notebooks folder, try parent directory
if Path.cwd().name == 'notebooks' or 'notebooks' in str(Path.cwd()):
    project_root = Path.cwd().parent
    config_paths.insert(0, project_root / 'config' / 'config.yaml')
    print(f"[Debug] Detected notebooks folder, also checking: {config_paths[0]}\n")

# Find config
config_path = None
for path in config_paths:
    if path.exists():
        config_path = path
        print(f"[OK] Found config at: {path}")
        break

if not config_path:
    print(f"[ERROR] Config not found in any of these locations:")
    for path in config_paths:
        print(f"  - {path}")
    print(f"\nCurrent working directory: {cwd}")
    print(f"Looking for: config/config.yaml")
    raise FileNotFoundError(f"Config file not found")

# Load configuration
with open(config_path) as f:
    config = yaml.safe_load(f)

print("\n[Config] Loaded configuration:")
print(f"  - Architecture: {config.get('model', {}).get('architecture', 'SSTZIP-GNN')}")
print(f"  - Methods: {config.get('clustering', {}).get('methods', [])}")
print(f"  - Time buckets: {config.get('temporal_aggregation', {}).get('buckets', [])}")
print(f"  - Batch size: {config.get('model', {}).get('training', {}).get('batch_size', 64)}")
print(f"  - Epochs: {config.get('model', {}).get('training', {}).get('epochs', 100)}")
print(f"  - Learning rate: {config.get('model', {}).get('training', {}).get('learning_rate', 0.001)}")

[Debug] Current working directory: /content/taxi-demand-prediction

[OK] Found config at: config/config.yaml

[Config] Loaded configuration:
  - Architecture: SSTZIP-GNN
  - Methods: ['method1', 'method2', 'method3']
  - Time buckets: [30]
  - Batch size: 128
  - Epochs: 10
  - Learning rate: 0.0003


In [ ]:
import sys
from pathlib import Path

# Add project to path
sys.path.insert(0, str(Path.cwd()))

# Import project modules
try:
    from src.data.data_loader import TaxiDemandDataModule
    from src.models.sstzip_gnn import SSTZIPGNNModel
    from src.training.trainer import SSTZIPGNNLightning
    from src.evaluation.metrics import Metrics
    print("[OK] All project modules imported successfully!")
except ImportError as e:
    print(f"[ERROR] Failed to import project modules: {e}")
    print("\nAvailable modules in src/:")
    import os
    for item in os.listdir('src'):
        print(f"  - {item}")
    raise

[OK] All project modules imported successfully!


In [ ]:
from pathlib import Path

print("[Check] Verifying required data files...\n")

# Check DuckDB features file
duckdb_path = Path('data/processed/taxi_features.duckdb')
if duckdb_path.exists():
    size_mb = duckdb_path.stat().st_size / 1e6
    print(f"  [OK] DuckDB: {duckdb_path} ({size_mb:.1f} MB)")
else:
    print(f"  [MISSING] DuckDB: {duckdb_path}")

# Check cluster assignment files
print("\n[Check] Cluster assignment files:")
cluster_files = {
    'Baseline': Path('data/models/baseline_clusters.pkl'),
    'Method1': Path('data/models/method1_clusters.pkl'),
    'Method2': Path('data/models/method2_clusters.pkl'),
    'Method3': Path('data/models/method3_clusters.pkl'),
}

missing_clusters = []
for name, path in cluster_files.items():
    if path.exists():
        print(f"  [OK] {name}: {path}")
    else:
        print(f"  [MISSING] {name}: {path}")
        missing_clusters.append(name)

if missing_clusters:
    print(f"\n[WARN] Missing cluster files for: {', '.join(missing_clusters)}")
    print("  These should be generated by Phase 3 (clustering scripts)")
else:
    print("\n[OK] All cluster files present!")

print("\n[Note] Data structure expected:")
print("  My Drive/data/")
print("    ├── processed/")
print("    │   ├── taxi_features.duckdb")
print("    │   ├── taxi_features_*.parquet")
print("    │   └── *_report.json")
print("    └── models/")
print("        └── sstzip_gnn/")
print("            ├── baseline_clusters.pkl")
print("            ├── method1_clusters.pkl")
print("            ├── method2_clusters.pkl")
print("            └── method3_clusters.pkl")

[Check] Verifying required data files...

  [OK] DuckDB: data/processed/taxi_features.duckdb (489.7 MB)

[Check] Cluster assignment files:
  [MISSING] Baseline: data/models/baseline_clusters.pkl
  [OK] Method1: data/models/method1_clusters.pkl
  [OK] Method2: data/models/method2_clusters.pkl
  [OK] Method3: data/models/method3_clusters.pkl

[WARN] Missing cluster files for: Baseline
  These should be generated by Phase 3 (clustering scripts)

[Note] Data structure expected:
  My Drive/data/
    ├── processed/
    │   ├── taxi_features.duckdb
    │   ├── taxi_features_*.parquet
    │   └── *_report.json
    └── models/
        └── sstzip_gnn/
            ├── baseline_clusters.pkl
            ├── method1_clusters.pkl
            ├── method2_clusters.pkl
            └── method3_clusters.pkl


In [ ]:
# Execute the stable training script
print("="*80)
print("EXECUTING: scripts/04_train_model_stable.py")
print("="*80)
print()

import os

script_path = os.path.join(os.getcwd(), 'scripts', '04_train_model_stable.py')

with open(script_path, encoding='utf-8') as f:
    training_script = f.read()

# Set __file__ for the script namespace
exec_globals = {'__file__': script_path, '__name__': '__main__'}
exec(training_script, exec_globals)

EXECUTING: scripts/04_train_model_stable.py

[WARN] Ignoring extra arguments: -f /root/.local/share/jupyter/runtime/kernel-40fdfab5-e252-4d79-b934-4397855e074d.json

STABLE SSTZIP-GNN TRAINING
[Env] Device: cpu
[Env] CUDA available: False
[Env] Methods: ['method1', 'method2', 'method3']

DATA SOURCE SUMMARY
[Schema]
  date_str	TIMESTAMP
  time_bucket	TIMESTAMP
  zone_id	INTEGER
  date_day	INTEGER
  hour	INTEGER
  demand_count	BIGINT
  avg_fare	FLOAT
  sum_fare	FLOAT
  min_fare	FLOAT
  max_fare	FLOAT
  stddev_fare	FLOAT
  avg_distance	FLOAT
  median_distance	FLOAT
  p95_distance	FLOAT
  avg_passenger	FLOAT
  window_start	TIMESTAMP
  window_end	TIMESTAMP
  Borough	VARCHAR
  Zone	VARCHAR
  service_zone	VARCHAR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


[Null Summary]
  rows: 3513833.0
  zones: 261.0
  stddev_fare_nulls: 959933.0
  avg_fare_nulls: 0.0
  avg_distance_nulls: 0.0
  avg_passenger_nulls: 36423.0
  demand_nulls: 0.0

[Range Summary]
  min_date: 2019-01-01 00:00:00
  max_date: 2020-06-30 00:00:00
  min_bucket: 2019-01-01 00:00:00
  max_bucket: 2020-06-30 00:00:00
  min_demand: 1
  max_demand: 588
  mean_demand: 20.957120899029636

CLUSTER ARTIFACTS
[OK] method1: method1_clusters.pkl
  keys: ['all_silhouette_scores', 'cluster_analysis', 'cluster_centers', 'feature_names', 'optimal_k', 'silhouette_score', 'zone_to_cluster']
  zone_count: 261
  cluster_counts: {0: 193, 2: 30, 1: 26, 3: 12}
[OK] method2: method2_clusters.pkl
  keys: ['all_davies_bouldin_scores', 'all_silhouette_scores', 'cluster_analysis', 'cluster_centers', 'cluster_semantics', 'davies_bouldin_score', 'feature_names', 'feature_scaling', 'optimal_k', 'silhouette_score', 'zone_to_cluster']
  zone_count: 261
  cluster_counts: {0: 79, 2: 19, 1: 163}
[OK] method3: 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 3513833 feature records from 261 zones
Adjacency matrix created: torch.Size([264, 264])
Train: 2621211 sequences
Val: 545704 sequences
Test: 322829 sequences
[Smoke] x shape: (8, 96, 9)
[Smoke] x finite: True
[Smoke] y shape: (8,)
[Smoke] y finite: True
[Smoke] pi finite: True
[Smoke] lambda finite: True
[Smoke] loss finite: True
[Smoke] loss: 29.936216

TRAINING: method1


...

[Config] Sequence length: 96
[Config] Batch size: 128
[Config] Training samples: 2621211
[Config] Validation samples: 545704
[Config] Test samples: 322829
[Config] Feature columns: ['avg_fare', 'sum_fare', 'min_fare', 'max_fare', 'stddev_fare', 'avg_distance', 'median_distance', 'p95_distance', 'avg_passenger']
[Train] Starting fit...


┏━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name               ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model              │ SSTZIPGNNModel │ 20.9 K │ train │     0 │
│ 1 │ feature_projection │ Linear         │    320 │ train │     0 │
└───┴────────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 21.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.2 K                                                                                               
Total estimated model params size (MB): 0.085                                                                      
Modules in train mode: 43                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

In [ ]:
import json
from pathlib import Path

print("[Results] Loading training summary...\n")

# Check multiple possible summary locations
summary_paths = [
    Path('logs/experiment_results.json'),
    Path('checkpoints/training_summary.json'),
]

results = None
for path in summary_paths:
    if path.exists():
        with open(path) as f:
            results = json.load(f)
        print(f"[OK] Found results at: {path}\n")
        break

if results:
    print("="*80)
    print("TRAINING RESULTS")
    print("="*80)
    print(json.dumps(results, indent=2))
else:
    print("[INFO] No summary file found yet. Check logs/ directory for individual experiment results.")
    print("\nLooking for experiment results...")
    logs_dir = Path('logs')
    if logs_dir.exists():
        results_files = list(logs_dir.glob('*_results.json'))
        if results_files:
            print(f"\nFound {len(results_files)} result file(s):")
            for f in results_files:
                print(f"  - {f.name}")

In [ ]:
# Display results as table if available
if results:
    print("\n" + "-"*80)
    print("METRICS SUMMARY")
    print("-"*80)

    if isinstance(results, dict) and 'results' in results:
        # Summary format from training_summary.json
        print(f"{'Method':<15} {'MAE':<12} {'RMSE':<12} {'MAPE':<12} {'Time (s)':<12}")
        print("-"*63)
        for method, metrics in results['results'].items():
            mae = metrics.get('MAE', 0)
            rmse = metrics.get('RMSE', 0)
            mape = metrics.get('MAPE', 0)
            time_s = metrics.get('Training_Time_Sec', 0)
            print(f"{method:<15} {mae:<12.4f} {rmse:<12.4f} {mape:<12.4f} {time_s:<12.1f}")
    else:
        # Other format
        print(json.dumps(results, indent=2))

In [ ]:
if IN_COLAB:
    import shutil
    from pathlib import Path

    print("[Sync] Copying results to Google Drive...")

    # Create checkpoints folder in Google Drive if it doesn't exist
    drive_checkpoints = Path('/content/drive/MyDrive/checkpoints')
    drive_checkpoints.mkdir(parents=True, exist_ok=True)

    # Copy local checkpoints
    local_checkpoints = Path('checkpoints')
    if local_checkpoints.exists():
        for method_dir in local_checkpoints.glob('method*'):
            dst = drive_checkpoints / method_dir.name
            if dst.exists():
                shutil.rmtree(dst)
            try:
                shutil.copytree(method_dir, dst)
                print(f"  [OK] {method_dir.name} → Google Drive")
            except Exception as e:
                print(f"  [ERROR] {method_dir.name}: {e}")

    # Copy logs
    drive_logs = Path('/content/drive/MyDrive/logs')
    drive_logs.mkdir(parents=True, exist_ok=True)

    local_logs = Path('logs')
    if local_logs.exists():
        for log_file in local_logs.glob('*.json'):
            try:
                shutil.copy(log_file, drive_logs / log_file.name)
            except:
                pass

    print("\n[OK] Results synced to Google Drive")
else:
    print("[Info] Running locally - results saved to:")
    print("  - checkpoints/  (model checkpoints)")
    print("  - logs/         (training logs & results)")

In [ ]:
print("\n" + "="*80)
print("TRAINING WORKFLOW COMPLETE")
print("="*80)

print("\n[Results] Output files:")
print("  - Checkpoints: checkpoints/<method>/<epoch-val_loss>.pt")
print("  - Metrics:     logs/*.json")

print("\n[Next Steps]")
print("  1. Review training results above")
print("  2. Run Phase 5 evaluation script: scripts/09_evaluation.py")
print("  3. Run Phase 6 MongoDB setup: scripts/test_mongodb_connection.py")

if IN_COLAB:
    print("\n[Files] Access results in Google Drive:")
    print("  - Checkpoints: checkpoints/ folder")
    print("  - Logs:        logs/ folder")
else:
    print("\n[Files] All results are in the project directory")

print("\n✅ Training complete!")